In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown
from huggingface_hub import InferenceClient
import gradio as gr

In [ ]:
load_dotenv()
router=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
load_dotenv()
ollama=OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
hugging=InferenceClient(api_key='')


In [3]:
system_prompt="""
you are a helpful assistant.
you add comments to the given code.comments should explain each detail.a beginner in codeing should also be able to understand the code
by reading comments.
and don't make any changes in the code or logic.and the code must provide same result before and after of adding comments.
Comment type:
Explain logic
Explain functions
Explain complex lines
Add docstrings
Explain algorithm
Preserve code:
comments only, no logic modifications

"""

def user_prompt(code):
    return f"""
    add comments to the given code of any language.it must be simple a beginner should also  be able to understand.
    don't make any changes in code or logic.just add the comments.
    detect the given coding language and add comments as per its syntax.
    here is the code{code} add comments to it.
    and it must be executable after adding comments. 
    """

In [4]:
def message(code):
    return [
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt(code)}
    ]

In [8]:
deepseek="deepseek-ai/DeepSeek-V3-0324"
qwen='Qwen/Qwen3.5-9B'

In [5]:
def commentor(model,code):
    if model=='deepseek':
        response=hugging.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3-0324",
            messages=message(code),
        )
    elif model=='qwen':
        response=hugging.chat.completions.create(
                    model='Qwen/Qwen3.5-9B',
                    messages=message(code),
                    max_tokens=50000
        )
    else:
        response=ollama.chat.completions.create(
            model="llama3.2",
            messages=message(code)
        )
    code=response.choices[0].message.content
    return code

In [6]:
python="""
def calculate(x, y):
    result = x * 2
    if result > y:
        return result - y
    return y - result
"""

In [ ]:
print(commentor(deepseek,python))

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python=gr.Textbox(label="python",lines=30)
        com= gr.Textbox(label="commented code",lines=30)
    with gr.Row():
        models=gr.Dropdown(['deepseek','llama','qwen'],label="select a model",value='deepseek')
        convert=gr.Button("convert")
    convert.click(commentor,inputs=[models,python],outputs=[com])
ui.launch()
# it is different from previous multimodal one because of  convert 
# if it was submit like it will be like that
# that was a chat bot which requires history it doesnot 
ui.launch()